In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit, root_scalar
from sklearn.metrics import r2_score

In [4]:
def complexityN0(n_data, t_dijkstra, std_dijkstra, t_bmssp, std_bmssp):
    def model_dijkstra(n, c):
        return c * (n * np.log2(n))

    def model_bmssp(n, c):
        return c * (n * np.power(np.log2(n), 2/3))
    
    params_d, _ = curve_fit(model_dijkstra, n_data, t_dijkstra, bounds=(0, np.inf), sigma=std_dijkstra, absolute_sigma=True)
    params_b, _ = curve_fit(model_bmssp, n_data, t_bmssp, bounds=(0, np.inf), sigma=std_bmssp, absolute_sigma=True)

    c_d = params_d[0]
    c_b = params_b[0]

    if c_d == 0 or c_b == 0:
        return -1
    
    log_n_intersect = np.power(c_b / c_d, 3).item()

    if log_n_intersect < np.log2(n_data[-1]):
        log_n_intersect = -1
    
    r2_d = r2_score(t_dijkstra, model_dijkstra(n_data, c_d))
    r2_b = r2_score(t_bmssp, model_bmssp(n_data, c_b))

    return {    
        'intersect_power10': round(log_n_intersect * np.log10(2), 3), # convert to power10
        'dijkstra': {
            "r2": round(r2_d, 3),
            "c": c_d.item()
        },
        'bmssp': {
            "r2": round(r2_b, 3),
            "c": c_b.item()
        }
    }

In [5]:
import pandas as pd

results_list = []
for graph_type in ['H3', 'D3', 'USA']:
    df = pd.read_csv(f'../outputs/results_{graph_type}/performance_summary_table.csv')
    n_data = df['Número de Vértices'].values
    t_dijkstra = df['Tempo Dijkstra (ms)'].values
    std_dijkstra = df['Desvio Dijkstra (ms)'].values
    t_bmssp = df['Tempo BMSPP (ms)'].values
    std_bmssp = df['Desvio BMSPP (ms)'].values
    
    res_comp = complexityN0(n_data, t_dijkstra, std_dijkstra, t_bmssp, std_bmssp)

    results_list.append({
        'Graph': graph_type,
        'C Dijkstra': f"{res_comp['dijkstra']['c']:.2e}",
        'R^2 Dijkstra': f"{res_comp['dijkstra']['r2']:.3f}",
        'C BMSSP': f"{res_comp['bmssp']['c']:.2e}",
        'R^2 BMSSP': f"{res_comp['bmssp']['r2']:.3f}",
        'C Ratio': f"{float(res_comp['bmssp']['c']) / float(res_comp['dijkstra']['c']):.2f}",
        'n0_log10': res_comp['intersect_power10'],
    })

for grid_type in ['GridR', 'GridED']:
    for grid_variant in ['S', 'R']:
        graph_name = f"{grid_variant}{grid_type}"
        df = pd.read_csv(f'../outputs/results_{grid_type}/performance_summary_table.csv')
        df = df[df['Graph File'].str.contains(f'{graph_name}')].copy()
        n_data = df['Número de Vértices'].values
        t_dijkstra = df['Tempo Dijkstra (ms)'].values
        std_dijkstra = df['Desvio Dijkstra (ms)'].values
        t_bmssp = df['Tempo BMSPP (ms)'].values
        std_bmssp = df['Desvio BMSPP (ms)'].values
        
        res_comp = complexityN0(n_data, t_dijkstra, std_dijkstra, t_bmssp, std_bmssp)

        results_list.append({
            'Graph': graph_name,
            'C Dijkstra': f"{res_comp['dijkstra']['c']:.2e}",
            'R^2 Dijkstra': f"{res_comp['dijkstra']['r2']:.3f}",
            'C BMSSP': f"{res_comp['bmssp']['c']:.2e}",
            'R^2 BMSSP': f"{res_comp['bmssp']['r2']:.3f}",
            'C Ratio': f"{float(res_comp['bmssp']['c']) / float(res_comp['dijkstra']['c']):.2f}",
            'n0_log10': res_comp['intersect_power10'],
        })

summary_df = pd.DataFrame(results_list)
display(summary_df)

,Graph,C Dijkstra,R^2 Dijkstra,C BMSSP,R^2 BMSSP,C Ratio,n0_log10
0,H3,2.25e-05,0.979,1.96e-04,0.887,8.74,200.746
1,D3,2.46e-05,0.997,2.14e-04,0.936,8.69,197.564
2,USA,7.62e-06,0.975,7.63e-05,0.919,10.02,302.814
3,SGridR,1.52e-05,0.977,1.49e-04,0.988,9.85,287.512
4,RGridR,9.25e-06,0.997,7.05e-05,0.887,7.63,133.614
5,SGridED,6.63e-06,0.999,1.17e-04,0.999,17.72,1675.365
6,RGridED,6.07e-06,0.987,1.14e-04,0.999,18.77,1989.293


In [6]:
def format_scientific(value):
    if value == 0:
        return "0"
    exponent = int(np.floor(np.log10(abs(value))))
    mantissa = value / (10 ** exponent)
    return f"{mantissa:.2f} \\times 10^{{{exponent}}}"

latex_output = r"""\begin{table}[!htpb]
    \centering
    \caption{Complexity analysis: Dijkstra vs BMSSP-WC}
    \begin{tabular}{lccccccc}
        \hline
        \textbf{Graph} & \textbf{C Dijkstra} & $\mathbf{R^2}$ \textbf{Dijkstra} & \textbf{C BMSSP} & $\mathbf{R^2}$ \textbf{BMSSP} & \textbf{C Ratio} & \textbf{n0} \\
        \hline
"""

for _, row in summary_df.iterrows():
    c_dijkstra = row['C Dijkstra']
    c_bmssp = row['C BMSSP']
    
    c_dijkstra_formatted = format_scientific(float(c_dijkstra))
    c_bmssp_formatted = format_scientific(float(c_bmssp))
    
    latex_output += f"        {row['Graph']} & ${c_dijkstra_formatted}$ & ${row['R^2 Dijkstra']}$ & ${c_bmssp_formatted}$ & ${row['R^2 BMSSP']}$ & ${row['C Ratio']}$ & $10^{{{row['n0_log10']}}}$ \\\\\n"

latex_output += r"""        \hline
    \end{tabular}
    \label{tab:complexity_analysis}
\end{table}
"""

print(latex_output)

\begin{table}[!htpb]
    \centering
    \caption{Complexity analysis: Dijkstra vs BMSSP-WC}
    \begin{tabular}{lccccccc}
        \hline
        \textbf{Graph} & \textbf{C Dijkstra} & $\mathbf{R^2}$ \textbf{Dijkstra} & \textbf{C BMSSP} & $\mathbf{R^2}$ \textbf{BMSSP} & \textbf{C Ratio} & \textbf{n0} \\
        \hline
        H3 & $2.25 \times 10^{-5}$ & $0.979$ & $1.96 \times 10^{-4}$ & $0.887$ & $8.74$ & $10^{200.746}$ \\
        D3 & $2.46 \times 10^{-5}$ & $0.997$ & $2.14 \times 10^{-4}$ & $0.936$ & $8.69$ & $10^{197.564}$ \\
        USA & $7.62 \times 10^{-6}$ & $0.975$ & $7.63 \times 10^{-5}$ & $0.919$ & $10.02$ & $10^{302.814}$ \\
        SGridR & $1.52 \times 10^{-5}$ & $0.977$ & $1.49 \times 10^{-4}$ & $0.988$ & $9.85$ & $10^{287.512}$ \\
        RGridR & $9.25 \times 10^{-6}$ & $0.997$ & $7.05 \times 10^{-5}$ & $0.887$ & $7.63$ & $10^{133.614}$ \\
        SGridED & $6.63 \times 10^{-6}$ & $0.999$ & $1.17 \times 10^{-4}$ & $0.999$ & $17.72$ & $10^{1675.365}$ \\
        RGridED 

In [3]:
def run_bootstrap_analysis(n_data, t_d, std_d, t_b, std_b, n_iterations=1000):
    c_d_boots, c_b_boots, n0_boots = [], [], []
    indices = np.arange(len(n_data))
    
    def model_dijkstra(n, c):
        return c * (n * np.log2(n))

    def model_bmssp(n, c):
        return c * (n * np.power(np.log2(n), 2/3))

    for _ in range(n_iterations):
        boot_idx = np.random.choice(indices, size=len(indices), replace=True)
        
        p_d, _ = curve_fit(model_dijkstra, n_data[boot_idx], t_d[boot_idx], 
                            sigma=std_d[boot_idx], absolute_sigma=True, bounds=(0, np.inf))
        p_b, _ = curve_fit(model_bmssp, n_data[boot_idx], t_b[boot_idx], 
                            sigma=std_b[boot_idx], absolute_sigma=True, bounds=(0, np.inf))
        
        cd, cb = p_d[0], p_b[0]
        log10_n0 = np.power(cb / cd, 3) * np.log10(2)
        
        c_d_boots.append(cd)
        c_b_boots.append(cb)
        n0_boots.append(log10_n0)

    return {
        'c_d_ci': np.percentile(c_d_boots, [2.5, 97.5]),
        'c_b_ci': np.percentile(c_b_boots, [2.5, 97.5]),
        'n0_ci': np.percentile(n0_boots, [2.5, 97.5]),
        'c_d_mean': np.mean(c_d_boots),
        'c_b_mean': np.mean(c_b_boots),
        'n0_mean': np.mean(n0_boots)
    }

# 2. Main execution loop for all graphs
results_list = []

# List of single-file graph types
graph_types = ['H3', 'D3', 'USA']
# List of grid-based variants
grid_types = [('GridR', ['SGridR', 'RGridR']), ('GridED', ['SGridED', 'RGridED'])]

# Process Standard Graphs
for gt in graph_types:
    df = pd.read_csv(f'../outputs/results_{gt}/performance_summary_table.csv')
    res = run_bootstrap_analysis(df['Número de Vértices'].values, 
                                 df['Tempo Dijkstra (ms)'].values, df['Desvio Dijkstra (ms)'].values,
                                 df['Tempo BMSPP (ms)'].values, df['Desvio BMSPP (ms)'].values)
    
    results_list.append({
        'Graph': gt,
        'C Dijkstra': f"{res['c_d_mean']:.2e}",
        'C Dijkstra CI': f"[{res['c_d_ci'][0]:.2e}, {res['c_d_ci'][1]:.2e}]",
        'C BMSSP': f"{res['c_b_mean']:.2e}",
        'C BMSSP CI': f"[{res['c_b_ci'][0]:.2e}, {res['c_b_ci'][1]:.2e}]",
        'n0_log10': round(res['n0_mean'], 2),
        'n0_log10 CI': f"[{res['n0_ci'][0]:.1f}, {res['n0_ci'][1]:.1f}]"
    })

# Process Grid Graphs
for folder, variants in grid_types:
    base_df = pd.read_csv(f'../outputs/results_{folder}/performance_summary_table.csv')
    for var in variants:
        df = base_df[base_df['Graph File'].str.contains(var)].copy()
        res = run_bootstrap_analysis(df['Número de Vértices'].values, 
                                     df['Tempo Dijkstra (ms)'].values, df['Desvio Dijkstra (ms)'].values,
                                     df['Tempo BMSPP (ms)'].values, df['Desvio BMSPP (ms)'].values)
        
        results_list.append({
            'Graph': var,
            'C Dijkstra': f"{res['c_d_mean']:.2e}",
            'C Dijkstra CI': f"[{res['c_d_ci'][0]:.2e}, {res['c_d_ci'][1]:.2e}]",
            'C BMSSP': f"{res['c_b_mean']:.2e}",
            'C BMSSP CI': f"[{res['c_b_ci'][0]:.2e}, {res['c_b_ci'][1]:.2e}]",
            'n0_log10': round(res['n0_mean'], 2),
            'n0_log10 CI': f"[{res['n0_ci'][0]:.1f}, {res['n0_ci'][1]:.1f}]"
        })

summary_df = pd.DataFrame(results_list)
display(summary_df)

,Graph,C Dijkstra,C Dijkstra CI,C BMSSP,C BMSSP CI,n0_log10,n0_log10 CI
0,H3,2.06e-05,"[1.49e-05, 2.32e-05]",1.93e-04,"[1.54e-04, 2.23e-04]",271.42,"[129.4, 578.8]"
1,D3,2.38e-05,"[1.96e-05, 2.57e-05]",2.11e-04,"[1.52e-04, 2.66e-04]",211.88,"[118.0, 340.5]"
2,USA,7.59e-06,"[6.90e-06, 8.33e-06]",7.51e-05,"[6.35e-05, 8.69e-05]",296.57,"[184.2, 428.6]"
3,SGridR,1.49e-05,"[1.22e-05, 1.67e-05]",1.39e-04,"[9.57e-05, 1.64e-04]",248.43,"[127.1, 374.1]"
4,RGridR,8.36e-06,"[4.28e-06, 9.54e-06]",7.26e-05,"[5.16e-05, 9.72e-05]",255.22,"[66.7, 549.6]"
5,SGridED,6.55e-06,"[5.75e-06, 6.80e-06]",1.14e-04,"[8.74e-05, 1.22e-04]",1606.63,"[931.1, 2007.3]"
6,RGridED,5.81e-06,"[4.78e-06, 6.15e-06]",1.08e-04,"[7.84e-05, 1.17e-04]",2013.81,"[1094.9, 3332.3]"


In [ ]:
def to_latex_sci(val_str):
    if 'e' not in val_str: return f"${val_str}$"
    base, exp = val_str.split('e')
    return f"${base} \\times 10^{{{int(exp)}}}$"

def format_ci_sci(ci_str):
    parts = ci_str.strip('[]').split(', ')
    return f"{to_latex_sci(parts[0])}, {to_latex_sci(parts[1])}"

# Header
print("\\begin{table}[!htpb]\n    \\centering\n    \\caption{Threshold point estimation of Dijkstra and BMSSP: Mean [95% CI]}\n    \\\label{tab:threshold_estimation}\n    \\small\n    \\renewcommand{\\arraystretch}{2.0}\n    \\begin{tabular}{lccc}\n        \\toprule\n        \\textbf{Graph} & $\\mathbf{c_1}$ & $\\mathbf{c_2}$ & $\\mathbf{n_0}$ \\\\\n        \\midrule")

for r in results_list:
    # Use 'n0_log10' and 'n0_log10 CI' which are the actual keys in your results_list
    n0_parts = r['n0_log10 CI'].strip('[]').split(', ')
    
    # Rounding to 0 decimal places
    low_n0 = int(round(float(n0_parts[0])))
    high_n0 = int(round(float(n0_parts[1])))
    mean_n0 = int(round(float(r['n0_log10']))) 

    row = (
        f"        {r['Graph']} & "
        f"\\makecell{{{to_latex_sci(r['C Dijkstra'])} \\\\ \\footnotesize [{format_ci_sci(r['C Dijkstra CI'])}]}} & "
        f"\\makecell{{{to_latex_sci(r['C BMSSP'])} \\\\ \\footnotesize [{format_ci_sci(r['C BMSSP CI'])}]}} & "
        f"\\makecell{{$10^{{{mean_n0}}}$ \\\\ \\footnotesize [$10^{{{low_n0}}}, 10^{{{high_n0}}}$]}} \\\\"
    )
    print(row + "\n        \\hline")

print("        \\bottomrule\n    \\end{tabular}\n\\end{table}")

\begin{table}[!htpb]
    \centering
    \caption{Threshold point estimation of Dijkstra and BMSSP: Mean [95\% CI]}
    \\label{tab:threshold_estimation}
    \small
    \renewcommand{\arraystretch}{2.0}
    \begin{tabular}{lccc}
        \toprule
        \textbf{Graph} & $\mathbf{c_1}$ & $\mathbf{c_2}$ & $\mathbf{n_0}$ \\
        \midrule
        H3 & \makecell{$2.06 \times 10^{-5}$ \\ \footnotesize [$1.49 \times 10^{-5}$, $2.32 \times 10^{-5}$]} & \makecell{$1.93 \times 10^{-4}$ \\ \footnotesize [$1.54 \times 10^{-4}$, $2.23 \times 10^{-4}$]} & \makecell{$10^{271}$ \\ \footnotesize [$10^{129}, 10^{579}$]} \\
        \hline
        D3 & \makecell{$2.38 \times 10^{-5}$ \\ \footnotesize [$1.96 \times 10^{-5}$, $2.57 \times 10^{-5}$]} & \makecell{$2.11 \times 10^{-4}$ \\ \footnotesize [$1.52 \times 10^{-4}$, $2.66 \times 10^{-4}$]} & \makecell{$10^{212}$ \\ \footnotesize [$10^{118}, 10^{340}$]} \\
        \hline
        USA & \makecell{$7.59 \times 10^{-6}$ \\ \footnotesize [$6.90 \times 10

<>:11: SyntaxWarning: invalid escape sequence '\l'
<>:11: SyntaxWarning: invalid escape sequence '\l'
/tmp/ipykernel_7863/2738391389.py:11: SyntaxWarning: invalid escape sequence '\l'
  print("\\begin{table}[!htpb]\n    \\centering\n    \\caption{Threshold point estimation of Dijkstra and BMSSP: Mean [95\\% CI]}\n    \\\label{tab:threshold_estimation}\n    \\small\n    \\renewcommand{\\arraystretch}{2.0}\n    \\begin{tabular}{lccc}\n        \\toprule\n        \\textbf{Graph} & $\\mathbf{c_1}$ & $\\mathbf{c_2}$ & $\\mathbf{n_0}$ \\\\\n        \\midrule")
